In [1]:
import pyspark
import pandas as pd
from pyspark.sql import SparkSession
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

In [4]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/08 12:29:51 WARN Utils: Your hostname, codespaces-6b7a3d, resolves to a loopback address: 127.0.0.1; using 10.0.11.12 instead (on interface eth0)
26/03/08 12:29:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 12:29:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


create the spark executor 

In [1]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv

--2026-03-07 20:42:33--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-07T21%3A42%3A53Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-07T20%3A42%3A06Z&ske=2026-03-07T21%3A42%3A53Z&sks=b&skv=2018-11-09&sig=mhAH6J%2B%2B1QDdHVCzpl1JHjaWlaEdRCUp8kI3Htu%2B3TA%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MjkxOTc1NCwibmJmIjoxNzcyOTE2MTU0LCJwYX

In [6]:
!gzip -d fhvhv_tripdata_2021-01.csv.gz
!wc -l fhvhv_tripdata_2021-01.csv

In [5]:
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv
df_pandas = pd.read_csv('head.csv')
df_pandas.dtypes
spark.createDataFrame(df_pandas).schema

In [9]:
from pyspark.sql import types
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [10]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [14]:
df.head(10)

[Row(hvfhs_license_num='HV0005', dispatching_base_num='B02510', pickup_datetime=datetime.datetime(2021, 1, 3, 17, 17, 21), dropoff_datetime=datetime.datetime(2021, 1, 3, 17, 26, 18), PULocationID=255, DOLocationID=34, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02882', pickup_datetime=datetime.datetime(2021, 1, 5, 22, 14, 7), dropoff_datetime=datetime.datetime(2021, 1, 5, 22, 32, 28), PULocationID=189, DOLocationID=107, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02867', pickup_datetime=datetime.datetime(2021, 1, 2, 17, 59, 55), dropoff_datetime=datetime.datetime(2021, 1, 2, 18, 10, 39), PULocationID=88, DOLocationID=137, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02872', pickup_datetime=datetime.datetime(2021, 1, 2, 23, 57, 54), dropoff_datetime=datetime.datetime(2021, 1, 3, 0, 15, 48), PULocationID=238, DOLocationID=224, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02875', pick

In [34]:
df = df.repartition(24)

In [36]:
df.write.parquet('fhvhv/2021/01/')

[Stage 7:=================================================>         (5 + 1) / 6]

AnalysisException: [PATH_ALREADY_EXISTS] Path file:/workspaces/spark_workshop/notebooks/fhvhv/2021/01 already exists. Set mode as "overwrite" to overwrite the existing path. SQLSTATE: 42K04

In [12]:
df = spark.read.parquet('fhvhv/2021/01/')

In [13]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [20]:
df.select('pickup_datetime','dropoff_datetime')\
.filter(df.hvfhs_license_num == 'HV0003')\
.show()

+-------------------+-------------------+
|    pickup_datetime|   dropoff_datetime|
+-------------------+-------------------+
|2021-01-05 22:14:07|2021-01-05 22:32:28|
|2021-01-02 17:59:55|2021-01-02 18:10:39|
|2021-01-02 23:57:54|2021-01-03 00:15:48|
|2021-01-06 15:53:13|2021-01-06 16:07:07|
|2021-01-07 07:35:24|2021-01-07 07:55:49|
|2021-01-07 08:45:12|2021-01-07 08:51:17|
|2021-01-02 15:44:26|2021-01-02 16:10:50|
|2021-01-04 16:50:28|2021-01-04 16:57:43|
|2021-01-03 10:30:34|2021-01-03 10:44:53|
|2021-01-03 22:05:20|2021-01-03 22:27:55|
|2021-01-04 08:01:02|2021-01-04 08:33:27|
|2021-01-02 13:01:10|2021-01-02 13:08:11|
|2021-01-06 17:12:27|2021-01-06 17:46:56|
|2021-01-04 09:05:18|2021-01-04 09:27:50|
|2021-01-06 16:46:47|2021-01-06 17:50:24|
|2021-01-06 08:03:47|2021-01-06 08:17:43|
|2021-01-04 06:45:42|2021-01-04 06:55:01|
|2021-01-03 13:20:41|2021-01-03 13:31:11|
|2021-01-03 17:30:33|2021-01-03 17:45:19|
|2021-01-06 20:55:57|2021-01-06 21:02:01|
+-------------------+-------------

In [16]:
!head -n 10 head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,


In [21]:
from pyspark.sql import functions as F

In [29]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [32]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [33]:
df\
    .withColumn('pickup_date', F.to_date(df.pickup_datetime))\
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime))\
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()
                

[Stage 6:>                                                          (0 + 1) / 1]

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/9ce| 2021-01-03|  2021-01-03|         255|          34|
|  e/b42| 2021-01-05|  2021-01-05|         189|         107|
|  e/b33| 2021-01-02|  2021-01-02|          88|         137|
|  e/b38| 2021-01-02|  2021-01-03|         238|         224|
|  e/b3b| 2021-01-06|  2021-01-06|         169|         208|
|  e/b33| 2021-01-07|  2021-01-07|          75|          88|
|  e/acc| 2021-01-07|  2021-01-07|         210|         210|
|  e/acc| 2021-01-02|  2021-01-02|         243|          69|
|  e/b35| 2021-01-04|  2021-01-04|         250|         213|
|  s/b3d| 2021-01-03|  2021-01-03|          87|          79|
|  e/a39| 2021-01-03|  2021-01-03|          68|         181|
|  s/acd| 2021-01-04|  2021-01-04|          95|         236|
|  s/b13| 2021-01-02|  2021-01-02|         262|         236|
|  e/9ce| 2021-01-04|  2